## 🔴 LEVEL 3: HARD - Supply Chain & Logistics Dataset

### Business Context
Logistics network managing shipments across multiple regions with complex hierarchical geography and multi-stage fulfillment tracking.

### Column Descriptions
1. **ShipmentID** - Unique shipment identifier (SHIP000001-SHIP002000)
2. **Region** - Geographic region (North, South, East, West, Central)
3. **District** - Sub-region within region (N1-N3, S1-S3, E1-E2, W1-W3, C1-C2)
4. **Warehouse** - Distribution warehouse code (WH_ABC, WH_DEF, etc.)
5. **SKU** - Stock keeping unit / product code
6. **ProductCategory** - Category of goods (Electronics, Textiles, Food, Furniture, Cosmetics)
7. **QuantityOrdered** - Units ordered by customer (1-1000)
8. **QuantityShipped** - Units actually shipped (0-1000)
9. **QuantityDelivered** - Units received by customer (0-1000)
10. **OrderDate** - Customer order date
11. **ShipDate** - Actual shipment date
12. **DeliveryDate** - Delivery completion date
13. **CostPerUnit** - Unit cost (mixed NULL representations)
14. **CarrierCode** - Shipping provider (FDX, UPS, DHL, USPS, TNT)
15. **DeliveryStatus** - Current status (Delivered, In Transit, Delayed, Lost, Cancelled)
16. **Notes** - Additional comments

### 🔍 Data Quality Issues (HARD)

#### Issue #1: Cross-Column Logic Violations (5% of data)
- **Location**: Quantity columns (QuantityOrdered, QuantityShipped, QuantityDelivered)
- **Type**: QuantityDelivered > QuantityOrdered (impossible), QuantityShipped > QuantityOrdered
- **Real-world scenario**: Warehouse adjustments, undisclosed refunds, system record mismatches
- **Challenge**: Identify which column has the error, investigate business rules
- **Skills**: Multi-column validation, business rule enforcement, root cause analysis

```
Before: 
  ShipmentID  QuantityOrdered  QuantityShipped  QuantityDelivered
  SHIP0001    100              90               95                ✓ Valid
  SHIP0002    100              95               150               ✗ Invalid (delivered more than ordered)
  SHIP0003    100              120              100               ✗ Invalid (shipped more than ordered)

After: Investigate, flag for review, or correct based on business logic
```

#### Issue #2: Temporal Logic Violations (4% of data)
- **Location**: Date columns (OrderDate, ShipDate, DeliveryDate)
- **Type**: DeliveryDate < ShipDate (delivered before shipped), ShipDate < OrderDate (shipped before ordered)
- **Real-world scenario**: Data entry errors, timezone issues, timestamp glitches
- **Challenge**: Identify which date is wrong, may require external validation
- **Skills**: Temporal validation, sequence analysis, timestamp debugging

```
Before:
  ShipmentID  OrderDate     ShipDate      DeliveryDate  Status
  SHIP0001    2023-01-15    2023-01-16    2023-01-20    ✓ Valid
  SHIP0002    2023-02-01    2023-02-10    2023-01-25    ✗ Invalid (delivered before shipped)
  SHIP0003    2023-03-05    2023-03-02    2023-03-10    ✗ Invalid (shipped before ordered)

After: Correct or flag for investigation
```

#### Issue #3: Hierarchical Category Mismatch (3% of data)
- **Location**: Region and District columns
- **Type**: District codes don't match assigned region (e.g., Region='North' but District='S1')
- **Real-world scenario**: Data migration errors, incomplete updates, referential integrity issues
- **Challenge**: Validate hierarchical relationships, correct or flag
- **Skills**: Referential validation, hierarchy mapping, data consistency checks

```
Valid Hierarchy:
  Region      Valid Districts
  North       N1, N2, N3
  South       S1, S2, S3
  East        E1, E2
  West        W1, W2, W3
  Central     C1, C2

Before (Invalid):
  Region='North', District='S1'  ✗ S1 belongs to South, not North
  Region='South', District='E1'  ✗ E1 belongs to East, not South

After: Correct mismatches or flag for review
```

#### Issue #4: Inconsistent Categorical Encoding (6% of data)
- **Location**: CarrierCode column
- **Type**: Multiple representations of same carrier ('FDX', 'fedex', 'Fedex Express', 'FedEX')
- **Real-world scenario**: Data from multiple sources, manual entry variations, system migrations
- **Challenge**: Identify equivalent values and map to canonical form
- **Skills**: Pattern matching, fuzzy matching, standardization

```
Before: ['FDX', 'fedex', 'Fedex Express', 'FedEX', 'UPS', 'ups', 'Ups']
After:  ['FDX', 'FDX', 'FDX', 'FDX', 'UPS', 'UPS', 'UPS']
```

#### Issue #5: Inconsistent Status Values (5% of data)
- **Location**: DeliveryStatus column
- **Type**: Multiple representations ('Delivered', 'delivered', 'DELIVERED', 'Dlvrd')
- **Real-world scenario**: Different systems, manual abbreviations, legacy data
- **Challenge**: Standardize all variants
- **Skills**: Categorical standardization, abbreviation resolution

```
Before: ['Delivered', 'delivered', 'DELIVERED', 'Dlvrd', 'In Transit', 'IN TRANSIT']
After:  ['Delivered', 'Delivered', 'Delivered', 'Delivered', 'In Transit', 'In Transit']
```

#### Issue #6: Multiple NULL Representations (7% of data)
- **Location**: CostPerUnit column
- **Type**: np.nan, 'N/A', 0, negative values
- **Real-world scenario**: Missing cost data, zero-cost items, incorrect entries
- **Challenge**: Standardize NULLs, distinguish between 0 and missing
- **Skills**: NULL standardization, handling multiple missing data encodings

```
Before: [50.5, np.nan, 'N/A', 0, -25.5, 100.0]
After:  [50.5, np.nan, np.nan, 0, np.nan, 100.0]
        (or handle differently based on business rules)
```

#### Issue #7: Negative and Invalid Values (2% of data)
- **Location**: CostPerUnit column
- **Type**: Negative costs (-50, -200)
- **Real-world scenario**: Data entry errors, refunds, system glitches
- **Challenge**: Determine if negative indicates refund or error
- **Skills**: Value validation, business rule application, context analysis

```
Before: [50.0, 75.5, -50.0, 100.0, -200.0, 25.0]
After:  [50.0, 75.5, np.nan, 100.0, np.nan, 25.0]
        (or investigate if these represent refunds/credits)
```

#### Issue #8: Decimal Precision Inconsistency (4% of data)
- **Location**: CostPerUnit column
- **Type**: Mixed decimal places (50, 50.5, 50.55, 50.5555)
- **Real-world scenario**: Different calculation systems, rounding variations
- **Challenge**: Standardize decimal precision while maintaining accuracy
- **Skills**: Numeric formatting, rounding strategies, precision handling

```
Before: [50, 50.5, 50.55, 50.5555, 75.123456]
After:  [50.00, 50.50, 50.55, 50.56, 75.12]
```

#### Issue #9: Whitespace and Case Issues (3% of data)
- **Location**: ProductCategory column
- **Type**: Extra whitespace (' Electronics '), case variation ('electronics')
- **Real-world scenario**: Manual data entry, copy-paste errors
- **Challenge**: Clean whitespace and standardize case
- **Skills**: String cleaning, case normalization

```
Before: ['Electronics', ' Electronics ', 'electronics', ' Textiles', 'FOOD']
After:  ['Electronics', 'Electronics', 'Electronics', 'Textiles', 'Food']
```

#### Issue #10: Sparse/Missing Categories (sparse)
- **Location**: ProductCategory column
- **Type**: Empty strings, missing categories
- **Real-world scenario**: Unclassified products, data entry incomplete
- **Challenge**: Decide on handling strategy (fill, remove, investigate)
- **Skills**: Handling missing categories, imputation decisions

```
Before: ['Electronics', '', 'Food', 'Furniture', '', 'Cosmetics']
After:  ['Electronics', 'Unclassified', 'Food', 'Furniture', 'Unclassified', 'Cosmetics']
        (or investigate what category '' represents)
```

### 🎯 Key Cleaning Tasks
1. Validate cross-column quantity relationships
2. Correct temporal sequence violations (Order → Ship → Delivery)
3. Validate Region-District hierarchical relationships
4. Standardize carrier codes
5. Standardize delivery status values
6. Standardize NULL representations for costs
7. Handle negative and impossible values
8. Normalize decimal precision
9. Clean whitespace and case inconsistencies
10. Handle missing/empty categories
11. Document all business rule violations
12. Create data quality report

### 💡 Estimated Time: 6-10 hours

### 🎓 Advanced Skills Required
- Multi-column validation
- Temporal logic understanding
- Hierarchical data validation
- Root cause analysis
- Data quality metrics
- SQL or Python proficiency for complex operations
- Business domain knowledge

---

Let's follow the Industry Standard Order for Data Cleaning and Prepration.

### **Step 1: Data Ingestion**

let's read the data first.

In [1]:
import pandas as pd
import os
# For backup
import numpy as np

# File Path.
file_path: str = 'messy_datasets_(3 sheets).xlsx'

# File Validation.
if os.path.exists(file_path):
    try:
        df_logistics = pd.read_excel(io=file_path, sheet_name='Hard_Level')
        print(f'Successfully read the File! {file_path}')
    except Exception as e:
        print(f'Error! While reading {e}')
else:
    raise FileNotFoundError(f'{file_path} Not Found!')

Successfully read the File! messy_datasets_(3 sheets).xlsx


File has been read, let's go for the next step...

### **Step 2: Data Profilling and Inspection**

Analyzing preassigned datatypes by pandas, statistical overview for numericals columns (Unmask few anomalies)......

Using:
- df_info() method, and
- df_describe()

In [2]:
df_logistics.info() # let's check the preassigned values.

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ShipmentID         2000 non-null   str           
 1   Region             2000 non-null   str           
 2   District           2000 non-null   str           
 3   Warehouse          2000 non-null   str           
 4   SKU                2000 non-null   str           
 5   ProductCategory    1680 non-null   str           
 6   QuantityOrdered    2000 non-null   int64         
 7   QuantityShipped    2000 non-null   int64         
 8   QuantityDelivered  2000 non-null   int64         
 9   OrderDate          2000 non-null   datetime64[us]
 10  ShipDate           2000 non-null   datetime64[us]
 11  DeliveryDate       2000 non-null   datetime64[us]
 12  CostPerUnit        1907 non-null   float64       
 13  CarrierCode        2000 non-null   str           
 14  DeliveryStatus     

**_NOTE:_ Expected columns to be reassign Data Types** 

| _columns_ | _Expected Datatypes_|
|---------|--------------------|
| **Region**  | Category |
| **District**| Category |
| **PrductCategory**| Category |
| **CarrierCode** | Category|
| **DeliveryStatus** | Category|

Category data types are memory efficient and performs gropby operations lightning fast.


In [3]:
# Statistical Overview
df_logistics.describe()

,QuantityOrdered,QuantityShipped,QuantityDelivered,OrderDate,ShipDate,DeliveryDate,CostPerUnit
count,2000.00000,2000.000000,2000.000000,2000,2000,2000,1907.000000
mean,484.07250,501.075500,505.357000,2023-07-02 20:52:48,2023-07-06 19:40:04.800000,2023-06-30 02:26:52.800000,238.852648
min,1.00000,0.000000,0.000000,2023-01-01 00:00:00,2022-12-23 00:00:00,2022-12-10 00:00:00,-476.390615
25%,225.00000,250.000000,258.000000,2023-03-30 00:00:00,2023-04-11 00:00:00,2023-03-29 00:00:00,109.735445
50%,480.00000,499.000000,500.500000,2023-07-05 00:00:00,2023-07-07 12:00:00,2023-06-30 00:00:00,242.757844
75%,730.25000,748.000000,750.250000,2023-10-03 06:00:00,2023-10-03 00:00:00,2023-09-28 06:00:00,379.417957
max,999.00000,1037.000000,1092.000000,2024-01-01 00:00:00,2024-01-01 00:00:00,2024-01-01 00:00:00,499.978692
std,289.90252,289.464358,289.059676,NaN,NaN,NaN,161.941379


**The Outliers and Anomalies**

- Look at the `QuantityOrdered` column the maximum quantity ordered is 999 however inside `QuantityShipped` and `QuantityDelivered` contains the max number is 1037 and 1092 respectively, logically which is not possible.

- At the `OrderDate` column the most oldest date is **Jan 1 2023**, however on the other hand , the `ShipDate` and `DeliveryDate` contains, **Dec 23 2022** and **Dec 10 2022**, The time Travelers.

- `CostPerUnit` Contains -476 dollars.

The rest of outliers we'll found on further operations.

let's follow the order...

### **Step 3: Structural Standardization**

Strip empty white spaces around the columns, standardized them on a particular format, drop completely empty columns (used for human readability sometimes).

In [4]:
# strip the whitespaces.
df_logistics.columns = df_logistics.columns.str.strip()

# Since it's already standardized, skip and drop the completely empty columns.
df_logistics = df_logistics.dropna(axis=1, how='all')

df_logistics.columns

Index(['ShipmentID', 'Region', 'District', 'Warehouse', 'SKU',
       'ProductCategory', 'QuantityOrdered', 'QuantityShipped',
       'QuantityDelivered', 'OrderDate', 'ShipDate', 'DeliveryDate',
       'CostPerUnit', 'CarrierCode', 'DeliveryStatus', 'Notes'],
      dtype='str')

### **Step 4: Deduplication**

- Exact Dupes, and
- Partial Dupes.

In [5]:
"""
Exact Dupes
"""
df_logistics.duplicated().sum()
# We have zero exact dupelicates.

np.int64(0)

In [6]:
"""
Partial Dupes
"""

# let's check how many duplicates does each columns have.
spaces = 20
for col in df_logistics.columns:
    print(f'{col}{" "*(spaces-len(col))}{df_logistics[col].duplicated().sum()}')

ShipmentID          0
Region              1995
District            1987
Warehouse           1950
SKU                 1
ProductCategory     1992
QuantityOrdered     1133
QuantityShipped     1123
QuantityDelivered   1134
OrderDate           1634
ShipDate            1635
DeliveryDate        1633
CostPerUnit         138
CarrierCode         1990
DeliveryStatus      1991
Notes               1373


The ShipmentID is free from the duplicates ensuring Data Integrity, However let's check for the warehouse accidentally logs the exact same cusotmer order twice. The syste generates two completely different `ShipmentID`s, but the actual real-world event is a duplicate.

In [7]:
# For that I've combined the columns 'SKU', 'QuantityOrdered', 'OrderDate'
# to form a composit key and let's check
df_logistics.duplicated(subset=['SKU', 'QuantityOrdered', 'OrderDate']).sum()

# result: 0 there's not any such kind of accidents.

np.int64(0)

### **Step 5: Data Type Casting**

**_NOTE:_ Expected columns to be reassign Data Types** 

| _columns_ | _Expected Datatypes_|
|---------|--------------------|
| **Region**  | Category |
| **District**| Category |
| **PrductCategory**| Category |
| **CarrierCode** | Category|
| **DeliveryStatus** | Category|

Category data types are memory efficient and performs gropby operations lightning fast.


In [8]:
# Before converting those columns let's check for each columns unique values.

columsn_list: list = ['Region', 'District', 'ProductCategory', 'CarrierCode', 'DeliveryStatus']

for col in columsn_list:
    print(f'{col}\n{df_logistics[col].unique()}')

Region
<ArrowStringArray>
['North', 'South', 'West', 'Central', 'East']
Length: 5, dtype: str
District
<ArrowStringArray>
['N2', 'S2', 'W1', 'W2', 'S1', 'C1', 'E1', 'S3', 'E2', 'N3', 'C2', 'N1', 'W3']
Length: 13, dtype: str
ProductCategory
<ArrowStringArray>
[         'Food',      'Textiles',   'Electronics',     'Furniture',
             nan,     'Cosmetics',   'electronics', ' Electronics ']
Length: 8, dtype: str
CarrierCode
<ArrowStringArray>
[            '?',          'USPS',           'UPS',           'FDX',
           'TNT',           'DHL',       'unknown', 'Fedex Express',
         'FedEX',         'fedex']
Length: 10, dtype: str
DeliveryStatus
<ArrowStringArray>
[   'Delayed',  'Cancelled', 'In Transit',       'Lost',      'Dlvrd',
    'Pending',  'delivered',  'Delivered',  'DELIVERED']
Length: 9, dtype: str


we'll tackle those when we finished the issues.

#### Issue #1: Cross-Column Logic Violations (5% of data)
- **Location**: Quantity columns (QuantityOrdered, QuantityShipped, QuantityDelivered)
- **Type**: QuantityDelivered > QuantityOrdered (impossible), QuantityShipped > QuantityOrdered
- **Real-world scenario**: Warehouse adjustments, undisclosed refunds, system record mismatches
- **Challenge**: Identify which column has the error, investigate business rules
- **Skills**: Multi-column validation, business rule enforcement, root cause analysis

```
Before: 
  ShipmentID  QuantityOrdered  QuantityShipped  QuantityDelivered
  SHIP0001    100              90               95                ✓ Valid
  SHIP0002    100              95               150               ✗ Invalid (delivered more than ordered)
  SHIP0003    100              120              100               ✗ Invalid (shipped more than ordered)

After: Investigate, flag for review, or correct based on business logic
```

In [9]:
# Let's make the copy of the dataframe first.
df_log_clean = df_logistics.copy()

# We perform the issue one.
# Let's make a mask which fullfills the following conditions
# QuantitityShipped <= QuantityOrder & QunatityDelivered <= QuantityOrdered

mask1 = (df_log_clean['QuantityShipped'] > df_log_clean['QuantityOrdered']) & (df_log_clean['QuantityDelivered'] > df_log_clean['QuantityOrdered'])

# df_log_clean[mask1]

# To handle this let's create a whole new column `QuantityReview`, containing the values of 'Reviewed'
# And for those columns which contains anamoies we'll flag them `UnderReview`
df_log_clean['QuantityReview'] = 'Reviewed'

# Now relplace with 'Under Review' containing the condition.
df_log_clean.loc[mask1, 'QuantityReview'] = 'Under Review'

# Now let's create a new condition where 'Delivered Quantity' > 'QuantityShipped'
mask2 = df_log_clean['QuantityDelivered'] > df_log_clean['QuantityShipped']
# It is the size of 1008 columns a massive error.
df_log_clean.loc[mask2, 'QuantityReview'] = 'Under Review'

df_log_clean['QuantityReview'].value_counts()

QuantityReview
Under Review    1378
Reviewed         622
Name: count, dtype: int64

#### Issue #2: Temporal Logic Violations (4% of data)
- **Location**: Date columns (OrderDate, ShipDate, DeliveryDate)
- **Type**: DeliveryDate < ShipDate (delivered before shipped), ShipDate < OrderDate (shipped before ordered)
- **Real-world scenario**: Data entry errors, timezone issues, timestamp glitches
- **Challenge**: Identify which date is wrong, may require external validation
- **Skills**: Temporal validation, sequence analysis, timestamp debugging

```
Before:
  ShipmentID  OrderDate     ShipDate      DeliveryDate  Status
  SHIP0001    2023-01-15    2023-01-16    2023-01-20    ✓ Valid
  SHIP0002    2023-02-01    2023-02-10    2023-01-25    ✗ Invalid (delivered before shipped)
  SHIP0003    2023-03-05    2023-03-02    2023-03-10    ✗ Invalid (shipped before ordered)

After: Correct or flag for investigation
```

In [10]:
# Normalizing few DeliveryStatus value, like delivered, Dlvrd, Delivered, DELIVERED,
# Wrong entries, or entries by multiple person.
normalizing_dict: dict = {
    'Dlvrd' : 'Delivered',
    'delivered': 'Delivered',
    'DELIVERED' : 'Delivered',
}

df_log_clean['DeliveryStatus'] = df_log_clean['DeliveryStatus'].str.strip().replace(normalizing_dict)

# let's the dates column and try to find out any pattern there.
df_log_clean.loc[:, ['OrderDate', 'ShipDate', 'DeliveryDate', 'DeliveryStatus']].head(40)

# let's seperate the Time travelrs in terms Product shipment first.
timetravel_mask = df_log_clean['ShipDate'] < df_log_clean['OrderDate']
# let's create a new column called 'DateReview' and fill with reviewed.
df_log_clean['DateReview'] = 'Reviewed'
# Let's fill the time travelers to 'Under review' on the DateReview column,
# And on the 'ShipDate' and 'DeliveryDate' column to NaT, With respect the OrderDate
# column.
df_log_clean.loc[timetravel_mask, 'DateReview'] = 'Under Review'
df_log_clean[timetravel_mask]

df_log_clean.loc[timetravel_mask, ['ShipDate', 'DeliveryDate']] = pd.NaT

df_log_clean[df_log_clean['DateReview']=='Under Review']


,ShipmentID,Region,District,Warehouse,SKU,ProductCategory,QuantityOrdered,QuantityShipped,QuantityDelivered,OrderDate,ShipDate,DeliveryDate,CostPerUnit,CarrierCode,DeliveryStatus,Notes,QuantityReview,DateReview
1,SHIP000002,South,S2,WH_CJB,SKU-877017,Textiles,575,20,202,2023-08-23,NaT,NaT,47.467756,USPS,Cancelled,Note 1,Under Review,Under Review
3,SHIP000004,West,W2,WH_FEA,SKU-304752,Food,263,940,975,2023-10-27,NaT,NaT,452.262667,USPS,In Transit,NaN,Under Review,Under Review
5,SHIP000006,Central,C1,WH_GDD,SKU-796112,NaN,923,241,807,2023-07-16,NaT,NaT,165.839112,UPS,Delayed,Note 5,Under Review,Under Review
7,SHIP000008,South,S3,WH_FDB,SKU-809416,Electronics,286,105,518,2023-09-13,NaT,NaT,105.471327,FDX,Cancelled,NaN,Under Review,Under Review
8,SHIP000009,East,E2,WH_GGE,SKU-840618,Cosmetics,23,342,612,2023-10-16,NaT,NaT,264.282682,TNT,Delayed,NaN,Under Review,Under Review
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1991,SHIP001992,Central,C2,WH_GGE,SKU-650003,Electronics,156,688,457,2023-08-01,NaT,NaT,12.641557,UPS,Cancelled,Note 1991,Under Review,Under Review
1993,SHIP001994,Central,C1,WH_DAB,SKU-340260,Food,855,74,252,2023-08-06,NaT,NaT,347.247478,?,Lost,Note 1993,Under Review,Under Review
1994,SHIP001995,East,E1,WH_FDB,SKU-551202,Electronics,617,864,363,2023-05-04,NaT,NaT,417.173639,unknown,Pending,Note 1994,Reviewed,Under Review
1995,SHIP001996,South,S3,WH_FEA,SKU-436682,Textiles,159,560,192,2023-10-27,NaT,NaT,170.801422,Fedex Express,Delayed,NaN,Under Review,Under Review


In [11]:
# Now let's handle those TimeTraveler with who got delivered before even placing the order.
delivery_timetravelers = df_log_clean['DeliveryDate'] < df_log_clean['OrderDate']

# let's fill those date with NaT and DeliveryReview to 'UnderReview'.
df_log_clean.loc[delivery_timetravelers, 'DateReview'] = 'Under Review'

# let's fill those date with NaT.
df_log_clean.loc[delivery_timetravelers, 'DeliveryDate'] = pd.NaT

In [12]:
# the only condition left where DeliveryDate < ShipDate.
# I'll see the Delivery Date, sinc according to business logic, delivery date,
# has much weight than ShipDate.

ship_timetravelers = df_log_clean['DeliveryDate']< df_log_clean['ShipDate']

# df_log_clean[ship_timetravelers]

# let's do the same flag them first Under Review, and replace them with NaT.
df_log_clean.loc[ship_timetravelers, 'DateReview'] = 'Under Review'

# Replace them with NaT.
df_log_clean.loc[ship_timetravelers, ['DeliveryDate', 'ShipDate']] = pd.NaT

df_log_clean[df_log_clean['DateReview']=='Under Review'].shape

# the only thing remains, is that 

(1654, 18)

In [13]:
# since we done that, now let's change the DeliveryStatus columns
df_log_clean['DeliveryStatus'].unique()

# For those columns which have NaT in DeliverDate It can be filled with any of these
# 'In Transit', 'Delayed', 'Pending', 'Lost' let's check the valid one first.

# Those columns which have ShipDate but not DeliveryDate.
undelivered_mask = df_log_clean['ShipDate'].notna() & df_log_clean['DeliveryDate'].isna()

# Only Those columns which have corrupted Status.
corrupted_status = undelivered_mask & df_log_clean['DeliveryStatus'].isin(['Cancelled', 'Delivered'])

# let's fill those columns with Pending... since this is the correct logic.
df_log_clean.loc[corrupted_status, 'DeliveryStatus'] = 'Pending'

In [14]:
# Now, we preserved the data integrity, by replacing in those status which was
# false at business logic, let's go for those rows which neither have ShipDate 
# nor DeliveryDate. Because of that physical reality, for an order, The Status
# pending, Delayed, or legitimately 'Cancelled' make parfect sense and should be
# left exactly as they are, 

# The only ones, that is physically Impossible ( and there for corrupted) are 
# 'Delayed', 'Lost', and 'In Transit'.

impossible_status = ['Lost', 'In Transit', 'Delivered']
cancelled_mask = df_log_clean['ShipDate'].isna() & df_log_clean['DeliveryDate'].isna()

# However let's check for corrupted_Statuses.
corrupted_statuses = cancelled_mask & df_log_clean['DeliveryStatus'].isin(impossible_status)

# only convert those which has impossible status.
df_log_clean.loc[corrupted_statuses, 'DeliveryStatus'] = 'Cancelled'

In [15]:
# A minor fix remains, all the columns which contains ShipDate and DeliveryDate,
# logically they should be stated Delivered, any other would be false status.

delivered_mask = df_log_clean['ShipDate'].notna() & df_log_clean['DeliveryDate'].notna()
# those all should be delivered.
df_log_clean.loc[delivered_mask, 'DeliveryStatus'] = 'Delivered'

In [16]:
df_log_clean[df_log_clean['DeliveryStatus']== 'Cancelled'].head()

,ShipmentID,Region,District,Warehouse,SKU,ProductCategory,QuantityOrdered,QuantityShipped,QuantityDelivered,OrderDate,ShipDate,DeliveryDate,CostPerUnit,CarrierCode,DeliveryStatus,Notes,QuantityReview,DateReview
1,SHIP000002,South,S2,WH_CJB,SKU-877017,Textiles,575,20,202,2023-08-23,NaT,NaT,47.467756,USPS,Cancelled,Note 1,Under Review,Under Review
3,SHIP000004,West,W2,WH_FEA,SKU-304752,Food,263,940,975,2023-10-27,NaT,NaT,452.262667,USPS,Cancelled,NaN,Under Review,Under Review
4,SHIP000005,South,S1,WH_IGJ,SKU-673386,Furniture,767,631,910,2023-01-15,NaT,NaT,228.925428,?,Cancelled,NaN,Under Review,Under Review
7,SHIP000008,South,S3,WH_FDB,SKU-809416,Electronics,286,105,518,2023-09-13,NaT,NaT,105.471327,FDX,Cancelled,NaN,Under Review,Under Review
9,SHIP000010,West,W2,WH_JBA,SKU-803534,Food,596,642,295,2023-07-13,NaT,NaT,336.413763,UPS,Cancelled,NaN,Reviewed,Under Review


In [17]:
# df_log_clean = df_log_clean.drop('DeliverStatus', axis=1)

# The final step is to convert the DeliveryStatus, QuantityReview and DateReview,
# columns to the category Dtype.

type_alter_col = ['DeliveryStatus', 'DateReview', 'QuantityReview']

df_log_clean = df_log_clean.astype({'DeliveryStatus': 'category', 'DateReview': 'category', 'QuantityReview': 'category'})

#### Issue #3: Hierarchical Category Mismatch (3% of data)
- **Location**: Region and District columns
- **Type**: District codes don't match assigned region (e.g., Region='North' but District='S1')
- **Real-world scenario**: Data migration errors, incomplete updates, referential integrity issues
- **Challenge**: Validate hierarchical relationships, correct or flag
- **Skills**: Referential validation, hierarchy mapping, data consistency checks

```
Valid Hierarchy:
  Region      Valid Districts
  North       N1, N2, N3
  South       S1, S2, S3
  East        E1, E2
  West        W1, W2, W3
  Central     C1, C2

Before (Invalid):
  Region='North', District='S1'  ✗ S1 belongs to South, not North
  Region='South', District='E1'  ✗ E1 belongs to East, not South

After: Correct mismatches or flag for review
```

**Which column Should we use as the key column and fix our data accordingly?**

- Region

- District

A _Region_ ('North') is a broad, general category. A _District_ ('S1') is a highly specific, granular location -- like a specific local warehouse or postal zone.

If a package's shipping label has a highly specific local barcode for District 'S1', but the general region dropdown menu says 'North', Which piece of information is more likely to represent the tru, physical location of that package? Well it's District, So we will use **District** as our key column in this particular scenario.

In [28]:
# let's create a valid dictionary.
valid_dis_dict: dict = {
    'North': ['N1', 'N2', 'N3'],
    'South': ['S1', 'S2', 'S3'],
    'East': ['E1', 'E2'],
    'West': ['W1', 'W2', 'W3'],
    'Central': ['C1', 'C2'],
}

# There's no need to valide dictionary bla bla bla.......
# let's flag those row which have corrupted Region-District values.
# Well I'll choose a creative approach specifically for this dataset.

# first let's check does 'Region' and 'District' column contains any Null values.
# df_log_clean.info() # I'm seeing that Those both column isn't contains any Null values.

"""
The approach: Not a very practical one, but I do like to find patterns inside the dataset I'm working on and adjust my code accordingly.

let's flag those rows whose Region's first word e.g. North = N isn't equal to it's District's first word e.g. W1 = W , we'll this approach
definitely flag those corrupted columns. And since the Districts contains high weightage over the Region columns, so we may say that let's
flag out the corrupted regions.
"""
corrupted_regions = df_log_clean['Region'].str[0] != df_log_clean['District'].str[0]
# let's find those rows
df_log_clean[corrupted_regions].head()

,ShipmentID,Region,District,Warehouse,SKU,ProductCategory,QuantityOrdered,QuantityShipped,QuantityDelivered,OrderDate,ShipDate,DeliveryDate,CostPerUnit,CarrierCode,DeliveryStatus,Notes,QuantityReview,DateReview
43,SHIP000044,South,E2,WH_HHG,SKU-466169,Furniture,810,184,350,2023-08-11,NaT,NaT,228.517249,DHL,Delayed,NaN,Under Review,Under Review
48,SHIP000049,South,N1,WH_HJA,SKU-242697,Cosmetics,563,168,705,2023-11-04,2023-12-10,2023-12-14,20.518341,USPS,Delivered,NaN,Under Review,Reviewed
67,SHIP000068,Central,N2,WH_DHB,SKU-763499,NaN,532,806,486,2023-07-03,NaT,NaT,461.025538,FDX,Cancelled,NaN,Reviewed,Under Review
80,SHIP000081,North,C1,WH_CIA,SKU-253499,Electronics,8,340,502,2023-03-06,2023-04-29,2023-12-07,408.567499,USPS,Delivered,NaN,Under Review,Reviewed
110,SHIP000111,South,E1,WH_FCH,SKU-422634,Electronics,89,294,211,2023-04-26,NaT,NaT,78.465108,FDX,Pending,Note 110,Under Review,Under Review


In [ ]:
# Now the fix: 

# let's create the dictionary first.
region_dic: dict = {
    'N': 'North',
    'S': 'South',
    'E': 'East',
    'W': 'West',
    'C': 'Central',
}

df_log_clean.loc[corrupted_regions, 'Region'] = df_log_clean['District'].str[0].map(region_dic)

df_log_clean.loc[corrupted_status, ['Region', 'District']] # viola fixed.....

,Region,District
2,West,W1
44,Central,C2
47,South,S2
51,Central,C2
53,East,E1
...,...,...
1953,East,E2
1955,East,E2
1974,East,E2
1981,South,S1


#### Issue #4: Inconsistent Categorical Encoding (6% of data)
- **Location**: CarrierCode column
- **Type**: Multiple representations of same carrier ('FDX', 'fedex', 'Fedex Express', 'FedEX')
- **Real-world scenario**: Data from multiple sources, manual entry variations, system migrations
- **Challenge**: Identify equivalent values and map to canonical form
- **Skills**: Pattern matching, fuzzy matching, standardization

```
Before: ['FDX', 'fedex', 'Fedex Express', 'FedEX', 'UPS', 'ups', 'Ups']
After:  ['FDX', 'FDX', 'FDX', 'FDX', 'UPS', 'UPS', 'UPS']